In [1]:
from transformers import AutoModel, AutoTokenizer
import torch
import os
import time

/home/lu.dong1/.conda/envs/deepseek-ocr2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import torch
import time

# ✅ 检测设备
if not torch.cuda.is_available():
    print("未检测到 GPU，将使用 CPU 运行")
    device = torch.device("cpu")
else:
    device = torch.device("cuda")
    print(f"使用 GPU: {torch.cuda.get_device_name(0)}")

# ⚙️ 调整此参数控制负载大小（越大负载越高）
SIZE = 4096

a = torch.randn(SIZE, SIZE, device=device)
b = torch.randn(SIZE, SIZE, device=device)

print("开始 GPU 压力循环...")
print("点击 Jupyter 工具栏的 ■ Stop 按钮，或按 I I（连按两次 I）可中断\n")

iteration = 0
start_time = time.time()

try:
    while True:
        c = torch.matmul(a, b)
        c = torch.relu(c)
        c = torch.matmul(c, a)
        c = torch.sigmoid(c)

        if device.type == "cuda":
            torch.cuda.synchronize()

        iteration += 1
        elapsed = time.time() - start_time

        if iteration % 10 == 0:
            print(f"迭代: {iteration:6d} | 耗时: {elapsed:.1f}s | 速率: {iteration/elapsed:.2f} iter/s")

except KeyboardInterrupt:
    print(f"\n✅ 已停止。共完成 {iteration} 次迭代，总耗时 {time.time()-start_time:.1f} 秒。")

使用 GPU: NVIDIA A100-SXM4-80GB
开始 GPU 压力循环...
点击 Jupyter 工具栏的 ■ Stop 按钮，或按 I I（连按两次 I）可中断

迭代:     10 | 耗时: 0.3s | 速率: 32.72 iter/s
迭代:     20 | 耗时: 0.5s | 速率: 43.96 iter/s
迭代:     30 | 耗时: 0.6s | 速率: 49.90 iter/s
迭代:     40 | 耗时: 0.7s | 速率: 53.50 iter/s
迭代:     50 | 耗时: 0.9s | 速率: 55.93 iter/s
迭代:     60 | 耗时: 1.0s | 速率: 57.67 iter/s
迭代:     70 | 耗时: 1.2s | 速率: 58.99 iter/s
迭代:     80 | 耗时: 1.3s | 速率: 60.01 iter/s
迭代:     90 | 耗时: 1.5s | 速率: 60.83 iter/s
迭代:    100 | 耗时: 1.6s | 速率: 61.51 iter/s
迭代:    110 | 耗时: 1.8s | 速率: 62.07 iter/s
迭代:    120 | 耗时: 1.9s | 速率: 62.55 iter/s
迭代:    130 | 耗时: 2.1s | 速率: 62.96 iter/s
迭代:    140 | 耗时: 2.2s | 速率: 63.31 iter/s
迭代:    150 | 耗时: 2.4s | 速率: 63.62 iter/s
迭代:    160 | 耗时: 2.5s | 速率: 63.90 iter/s
迭代:    170 | 耗时: 2.7s | 速率: 64.14 iter/s
迭代:    180 | 耗时: 2.8s | 速率: 64.36 iter/s
迭代:    190 | 耗时: 2.9s | 速率: 64.56 iter/s
迭代:    200 | 耗时: 3.1s | 速率: 64.73 iter/s
迭代:    210 | 耗时: 3.2s | 速率: 64.90 iter/s
迭代:    220 | 耗时: 3.4s | 速率: 65.05 iter/s
迭代:    2

In [2]:
image_path = "data/temp/X00016469612.jpg"

In [3]:
model_name = "deepseek-ai/DeepSeek-OCR-2"

In [4]:
output_dir = "data/temp/deepseek_ocr2_output"
os.makedirs(output_dir, exist_ok=True)

In [13]:
grouding_mode = os.path.join(output_dir, "grounding")
markdown_mode = os.path.join(output_dir, "markdown")

In [6]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

In [7]:
model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True,
    use_safetensors=True,
)

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.


In [8]:
model = model.eval().cuda().to(torch.bfloat16)

In [11]:
prompt_ground = "<image>\n<|grounding|>Convert the document to markdown."
prompt_md = "<image>\nConvert the document to markdown."

In [14]:
torch.cuda.synchronize()
start = time.perf_counter()

res = model.infer(
    tokenizer,
    prompt=prompt_ground,
    image_file=image_path,
    output_path=grouding_mode,
    base_size=1024,
    image_size=768,
    crop_mode=True,
    save_results=True,
)

torch.cuda.synchronize()
end = time.perf_counter()

print(res)
print(f"Elapsed time: {end - start:.2f} seconds")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([2, 144, 1280])
<|ref|>sub_title<|/ref|><|det|>[[150, 25, 700, 64]]<|/det|>
## tan woon yann

<|ref|>text<|/ref|><|det|>[[144, 87, 916, 112]]<|/det|>
BOOK TA_K (TAMAN DAYA) SDN BHD

<|ref|>text<|/ref|><|det|>[[440, 114, 625, 137]]<|/det|>
789+17-W

<|ref|>text<|/ref|><|det|>[[227, 138, 830, 162]]<|/det|>
NO.5: 55,57 & 59, JALAN SAGU 18,

<|ref|>text<|/ref|><|det|>[[410, 163, 655, 185]]<|/det|>
TAMAN DAYA,

<|ref|>text<|/ref|><|det|>[[346, 186, 730, 209]]<|/det|>
81100 JOHOR BAHRU,

<|ref|>text<|/ref|><|det|>[[463, 210, 600, 231]]<|/det|>
JOHOR.

<|ref|>image<|/ref|><|det|>[[189, 264, 864, 320]]<|/det|>


<|ref|>text<|/ref|><|det|>[[100, 332, 610, 356]]<|/det|>
Document No : TD01167104

<|ref|>text<|/ref|><|det|>[[100, 362, 751, 386]]<|/det|>
Date : 25/12/2018 8:13:39 PM

<|ref|>text<|/ref|><|det|>[[100, 388, 475, 410]]<|/det|>
Cashier : MANIS

<|ref|>text<|/ref|><|det|>[[100, 413, 275, 435]]<|/det|>
Member :

<|ref|>sub_title<|/ref

other: 100%|██████████| 18/18 [00:00<00:00, 304425.29it/s]

None
Elapsed time: 16.29 seconds


In [15]:
torch.cuda.synchronize()
start = time.perf_counter()

res = model.infer(
    tokenizer,
    prompt=prompt_md,
    image_file=image_path,
    output_path=markdown_mode,
    base_size=1024,
    image_size=768,
    crop_mode=True,
    save_results=True,
)

torch.cuda.synchronize()
end = time.perf_counter()

print(res)
print(f"Elapsed time: {end - start:.2f} seconds")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([2, 144, 1280])
# tan moon yann

**BOOK TA_K (TAMAN DAYA) SDN BHD**  
789-17-W  
NO.5: 55,57 & 59, JALAN SAGU 18,  
TAMAN DAYA,  
81100 JOHOR BAHRU,  
JOHOR.  

---

**Document No** : TD01167104  
**Date** : 25/12/2018 8:13:39 PM  
**Cashier** : MANIS  
**Member** :  

---

# CASH BILL

| CODE/DESC    | PRICE | DISC | AMOUNT |
|---|---|---|---|
| QTY    |    |    | RM    |
| 95569390-0118    | *    |    |    |

**F. MODELING CLAY KIDDY FISH**  
1 PC  
9.00  0.00  9.00  

**Total**:  9.00  
**Rour Jing Adjustment**: 0.00  

**Round : d Total (RM):**  
9.00  

---

**Cash**  20.00  
**CHANGE**  10.00  

---

**9.00**  

---

**GOODS SOLD ARE NOT REFURBABLE OR EXCHANGEABLE**  
(说明：此页无此栏可退)  

---

**THANK YOU**  
PLEASE COME AGAIN !
===============save results:===============


image: 0it [00:00, ?it/s]
other: 0it [00:00, ?it/s]

None
Elapsed time: 7.71 seconds


# V3

In [1]:
import os
import re
import io
import json
import contextlib

from transformers import AutoModel, AutoTokenizer
import torch

/home/lu.dong1/.conda/envs/deepseek-ocr2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
image_path = "data/temp/X00016469612.jpg"
model_name = "deepseek-ai/DeepSeek-OCR-2"
output_dir = "data/temp/deepseek_ocr2_output"
os.makedirs(output_dir, exist_ok=True)

In [3]:
doc_id = os.path.basename(image_path).split(".")[0]
raw_ocr_path = f"{output_dir}/{doc_id}_raw_ocr.txt"
raw_text_path = f"{output_dir}/{doc_id}_raw_text.txt"
range_json_path = f"{output_dir}/{doc_id}_ranges.json"

In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True,
    use_safetensors=True,
)

model = model.eval().cuda().to(torch.bfloat16)

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.


In [5]:
def capture_infer(prompt):

    buffer = io.StringIO()

    with contextlib.redirect_stdout(buffer):
        model.infer(
            tokenizer,
            prompt=prompt,
            image_file=image_path,
            output_path=output_dir,
            base_size=1024,
            image_size=768,
            crop_mode=True,
            save_results=False
        )

    return buffer.getvalue()

In [6]:
def remove_logs(text):

    lines = text.splitlines()
    cleaned = []

    for line in lines:
        s = line.strip()

        if s.startswith("BASE:"):
            continue
        if s.startswith("PATCHES:"):
            continue
        if s.startswith("===="):
            continue

        cleaned.append(line)

    return "\n".join(cleaned).strip()

In [15]:
def markdown_to_clean_text(md):
    text = md

    text = re.sub(r'^\s{0,3}#{1,6}\s*', '', text, flags=re.MULTILINE)
    text = re.sub(r'(\*\*|\*|__|_|`)', '', text)
    text = re.sub(r'^\s*[-]{3,}\s*$', '', text, flags=re.MULTILINE)

    lines = text.splitlines()
    out = []

    for line in lines:

        s = line.strip()

        if not s:
            continue

        if s.startswith("![]("):
            continue

        if "|" in s:
            parts = [p.strip() for p in s.strip("|").split("|")]
            parts = [p for p in parts if p]
            out.append(" ".join(parts))
            continue

        if "<table>" in s:
            continue

        out.append(s)

    return "\n".join(out)

In [8]:
def normalize(s):

    s = s.lower()
    s = s.replace("_", " ")
    s = re.sub(r'\s+', ' ', s)
    return s.strip()

In [9]:
def split_lines(text):

    return [l.strip() for l in text.splitlines() if l.strip()]

In [14]:
# ========= 1 markdown OCR =========

prompt_md = "<image>\nConvert the document to markdown."

md_output = capture_infer(prompt_md)
md_output = remove_logs(md_output)

with open(raw_ocr_path, "w", encoding="utf-8") as f:
    f.write(md_output)


# ========= 2 clean raw text =========

raw_text = markdown_to_clean_text(md_output)

with open(raw_text_path, "w", encoding="utf-8") as f:
    f.write(raw_text)

lines = split_lines(raw_text)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


In [11]:
# ========= 3 grounding =========

prompt_ground = "<image>\n<|grounding|>Convert the document to markdown."

ground_output = capture_infer(prompt_ground)
ground_output = remove_logs(ground_output)


# ========= 4 找 table block =========

table_pos = ground_output.find("<|ref|>table")

anchor_text = None

if table_pos != -1:

    before = ground_output[:table_pos]

    anchors = ["cash bill", "invoice", "receipt"]

    lines_g = before.splitlines()

    for l in reversed(lines_g):

        ln = normalize(l)

        for a in anchors:
            if a in ln:
                anchor_text = l
                break

        if anchor_text:
            break

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


In [12]:
# ========= 5 line mapping =========

anchor_line = None

if anchor_text:

    anchor_norm = normalize(anchor_text)

    for i, l in enumerate(lines):

        if anchor_norm in normalize(l):
            anchor_line = i
            break


# fallback: table header

if anchor_line is None:

    for i, l in enumerate(lines):

        if "code" in normalize(l) and "price" in normalize(l):
            anchor_line = i
            break

In [13]:
# ========= ranges =========

header_range = [0, anchor_line]
context_range = [anchor_line, len(lines)]


ranges = {
    "header_line_range": header_range,
    "context_line_range": context_range
}


with open(range_json_path, "w") as f:
    json.dump(ranges, f, indent=2)